# PRIDE ID Analysis in PubMed Central

Search for PRIDE proteomics identifiers (PXD/PRD) in full-text PMC articles and supplementary materials.

**Pipeline:**
1. Search PMC for article IDs by query
2. Fetch full-text XML and extract PRIDE identifiers
3. Batch processing with incremental CSV checkpointing
4. Year-by-year massive collection
5. Merge and deduplicate results
6. Verify and enrich via BioC API and supplementary archives

## 1. Imports & Configuration

In [1]:
import pandas as pd
import requests
import xml.etree.ElementTree as ET
import re
import io
import os
from dotenv import load_dotenv
import json
import time
import tarfile
import logging
import functools
from datetime import datetime
from collections import Counter
from IPython.display import display, HTML
from tqdm.auto import tqdm

In [2]:
load_dotenv()

config = {
    "term": "Ovarian cancer proteomics",
    "years": 5
}

API_KEY = os.getenv("NCBI_API_KEY")

PRIDE_PATTERN = re.compile(r'P[XR]D\d{6}', re.IGNORECASE)

## 2. Logging

In [5]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

class AnalysisLogger:
    def __init__(self, session_name="PrideAnalysis", log_to_file=True):
        self.logger = logging.getLogger(session_name)
        self.logger.setLevel(logging.DEBUG)
        self.logger.handlers = []
        formatter = logging.Formatter('%(asctime)s [%(levelname)s] %(message)s', datefmt='%H:%M:%S')
        console_handler = logging.StreamHandler()
        console_handler.setFormatter(formatter)
        self.logger.addHandler(console_handler)
        if log_to_file:
            if not os.path.exists('logs'):
                os.makedirs('logs')
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            file_handler = logging.FileHandler(f'logs/session_{timestamp}.log', encoding='utf-8')
            file_handler.setFormatter(formatter)
            self.logger.addHandler(file_handler)

    def set_debug(self, enabled):
        level = logging.DEBUG if enabled else logging.INFO
        for handler in self.logger.handlers:
            handler.setLevel(level)
        self.logger.setLevel(level)

session_logger = AnalysisLogger()
log = session_logger.logger

def trace_step(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        log.debug(f"Enter {func.__name__}")
        start_time = time.perf_counter()
        try:
            result = func(*args, **kwargs)
            elapsed = time.perf_counter() - start_time
            log.debug(f"{func.__name__} completed in {elapsed:.3f}s")
            return result
        except Exception as e:
            log.error(f"Error in {func.__name__}: {str(e)}", exc_info=True)
            raise
    return wrapper

## 3. Core API Functions

In [6]:
@trace_step
def fetch_pmc_ids(query, start_year, end_year, limit):
    """Search PMC for article IDs matching query and date range."""
    esearch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    params = {
        "db": "pmc",
        "term": query,
        "mindate": f"{start_year}/01/01",
        "maxdate": f"{end_year}/12/31",
        "datetype": "pdat",
        "retmode": "json",
        "retmax": limit
    }
    response = requests.get(esearch_url, params=params, timeout=100)
    response.raise_for_status()
    return response.json().get("esearchresult", {}).get("idlist", [])

In [ ]:
@trace_step
def fetch_xml_content(pmc_ids, api_key=None, max_retries=3):
    """Fetch full XML for a list of PMC IDs with retry logic."""
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    params = {
        "db": "pmc",
        "id": ",".join(map(str, pmc_ids)),
        "retmode": "xml",
        "api_key": api_key
    }
    for attempt in range(max_retries):
        try:
            response = requests.get(url, params=params, timeout=60)
            response.raise_for_status()
            return ET.fromstring(response.content)
        except (requests.exceptions.ChunkedEncodingError,
                requests.exceptions.ConnectionError,
                requests.exceptions.Timeout) as e:
            wait_time = (attempt + 1) * 5
            log.warning(f"Network error (attempt {attempt+1}/{max_retries}): {e}")
            time.sleep(wait_time)
        except Exception as e:
            log.error(f"Critical error fetching XML: {e}")
            break
    return None

In [7]:
@trace_step
def extract_article_info(article_node):
    """Extract PMCID and title from an article XML node."""
    pmcid_elem = article_node.find(".//article-id[@pub-id-type='pmc']")
    if pmcid_elem is not None:
        pmcid = pmcid_elem.text if pmcid_elem.text.startswith('PMC') else f"PMC{pmcid_elem.text}"
    else:
        pmid_node = article_node.find(".//article-id[@pub-id-type='pmid']")
        pmcid = f"PMID:{pmid_node.text}" if pmid_node is not None else "Unknown"
    title_node = article_node.find(".//article-title")
    title = "".join(title_node.itertext()).strip() if title_node is not None else "No Title"
    return pmcid, title

In [8]:
@trace_step
def find_pride_ids(article_node, pmcid):
    """Find PRIDE/PRD identifiers in article full text via XML itertext."""
    full_text = "".join(article_node.itertext())
    char_count = len(full_text)
    log.debug(f"Analyzing {pmcid}: {char_count} chars")
    matches = set(m.upper() for m in PRIDE_PATTERN.findall(full_text))
    if not matches and char_count < 5000:
        log.warning(f"{pmcid}: no matches found, but text is very short ({char_count} chars)")
    return matches, char_count

In [9]:
@trace_step
def extract_supplementary_info(article_node):
    """Extract file extensions from supplementary material metadata."""
    extensions = []
    for supp in article_node.findall(".//supplementary-material"):
        media = supp.find(".//media")
        if media is not None:
            href = media.get("{http://www.w3.org/1999/xlink}href", "")
            if href and "." in href:
                extensions.append(href.split('.')[-1].lower())
            else:
                subtype = media.get("mime-subtype", "")
                if subtype:
                    extensions.append(subtype.split('.')[-1].lower())
    counts = Counter(extensions)
    if not counts:
        return ""
    return ", ".join(f"{ext}({count})" for ext, count in counts.items())

## 4. Batch Processing

In [10]:
def process_single_batch(batch_ids, api_key, supp_files_enabled):
    """Process a single batch of PMC IDs: fetch XML, extract PRIDE IDs."""
    batch_results = []
    root = fetch_xml_content(batch_ids, api_key)
    if root is None:
        return []
    for article in root.findall(".//article"):
        pmcid, title = extract_article_info(article)
        pride_ids, char_count = find_pride_ids(article, pmcid)
        supp_str = "Skipped"
        if supp_files_enabled:
            supp_data = extract_supplementary_info(article)
            supp_str = supp_data if supp_data else "None"
        for p_id in pride_ids:
            batch_results.append({
                "PRIDE_ID": p_id,
                "PMCID": pmcid,
                "Title": title,
                "Chars": char_count,
                "Supp_Ext": supp_str
            })
    root.clear()
    return batch_results

## 6. Year-by-Year Massive Collection

Split search by year to bypass the 10,000 article API limit per request.

In [ ]:
def run_massive_analysis_by_years(query, api_key=None, batch_size=50):
    """Run batched analysis year by year to bypass the 10k article API limit."""
    years = [2022, 2023, 2024, 2025, 2026]
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file = f"massive_analysis_{timestamp}.csv"

    log.info(f"Starting year-by-year collection. Output: {output_file}")

    for year in years:
        log.info(f"Processing year {year}...")
        ids = fetch_pmc_ids(query, year, year, limit=50000)
        if not ids:
            log.warning(f"No articles found for {year}.")
            continue
        log.info(f"Found {len(ids)} articles for {year}. Starting batch processing...")

        for i in tqdm(range(0, len(ids), batch_size), desc=f"Year {year}"):
            batch = ids[i : i + batch_size]
            batch_data = process_single_batch(batch, api_key, supp_files_enabled=True)
            if batch_data:
                df_batch = pd.DataFrame(batch_data)
                file_exists = os.path.isfile(output_file)
                df_batch.to_csv(output_file, mode='a', index=False,
                               header=not file_exists, encoding='utf-8')
            time.sleep(0.2 if api_key else 0.6)

    if os.path.exists(output_file):
        return pd.read_csv(output_file)
    return None

In [ ]:
# final_df = run_massive_analysis_by_years("Ovarian cancer proteomics", api_key=API_KEY)

## 7. Merge & Deduplicate Results

In [ ]:
def merge_pride_files(files, output_file="merged_pride_data.csv"):
    """Merge multiple CSV/TSV result files, removing duplicates."""
    existing_files = [f for f in files if os.path.exists(f)]
    for f in files:
        log.info(f"  {'Found' if f in existing_files else 'Missing'}: {f}")

    if not existing_files:
        log.warning("No files available for merging!")
        return None

    dfs = []
    for f in existing_files:
        try:
            df = pd.read_csv(f)
        except Exception:
            df = pd.read_csv(f, sep='\t')
        dfs.append(df)
        log.info(f"  Loaded {len(df)} records from {f}")

    merged_df = pd.concat(dfs, ignore_index=True)
    initial_count = len(merged_df)
    merged_df = merged_df.drop_duplicates()
    log.info(f"Total records: {len(merged_df)} (removed {initial_count - len(merged_df)} duplicates)")
    log.info(f"Columns: {', '.join(merged_df.columns)}")

    merged_df.to_csv(output_file, index=False, encoding='utf-8-sig')
    log.info(f"Saved to: {output_file}")
    return merged_df

In [ ]:
def merge_with_duplicate_analysis(file1, file2, output_file="merged_pride_analysis.csv"):
    """Merge two CSV files with detailed duplicate analysis by PRIDE_ID + PMCID."""
    df1 = pd.read_csv(file1)
    df2 = pd.read_csv(file2)
    log.info(f"{file1}: {len(df1)} records")
    log.info(f"{file2}: {len(df2)} records")

    merged = pd.concat([df1, df2], ignore_index=True)
    unique_merged = merged.drop_duplicates(subset=['PRIDE_ID', 'PMCID'], keep='first')

    log.info(f"Total records: {len(merged)}")
    log.info(f"Unique PRIDE_ID+PMCID combinations: {len(unique_merged)}")
    log.info(f"Duplicates removed: {len(merged) - len(unique_merged)}")

    unique_merged.to_csv(output_file, index=False, encoding='utf-8-sig')
    log.info(f"Saved to: {output_file}")

    if 'PRIDE_ID' in unique_merged.columns:
        log.info("Top-10 PRIDE_ID by frequency:")
        for pride_id, count in unique_merged['PRIDE_ID'].value_counts().head(10).items():
            log.info(f"  {pride_id}: {count}")

    return unique_merged

## 8. BioC Verification

Re-verify results from the main pipeline using the BioC full-text API.
Searches for PRIDE IDs that may have been missed by the eFetch XML path.

> **Note:** NCBI OA supplementary archive API (`oa.fcgi`) was retired in August 2026.
> Verification is now BioC full-text only.

In [ ]:
def extract_pride_ids_from_content(content):
    """Extract PRIDE IDs from text or binary content."""
    if isinstance(content, bytes):
        try:
            content = content.decode('utf-8', errors='ignore')
        except Exception:
            content = str(content)
    return set(m.upper() for m in PRIDE_PATTERN.findall(content))

def get_pmcid_from_pmid(pmid):
    """Convert PMID to PMCID via NCBI ID Converter API.

    Note: endpoint moved to pmc.ncbi.nlm.nih.gov in 2026 and returns XML.
    """
    email = os.getenv("USER_EMAIL", "")
    url = (
        f"https://pmc.ncbi.nlm.nih.gov/tools/idconv/api/v1/articles/"
        f"?ids={pmid}&tool=pride_pmc_analysis&email={email}"
    )
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            root = ET.fromstring(r.content)
            record = root.find(".//record")
            if record is not None:
                return record.get("pmcid")
    except Exception:
        pass
    return None

def parse_pmcid_from_string(pmcid_str):
    """Parse PMCID from various formats: PMID:123, PMC123, or bare numeric ID."""
    if pd.isna(pmcid_str) or not pmcid_str or str(pmcid_str).strip() == '':
        return None
    pmcid_str = str(pmcid_str).strip()
    if pmcid_str.startswith('PMID:'):
        return get_pmcid_from_pmid(pmcid_str.replace('PMID:', ''))
    if pmcid_str.startswith('PMC'):
        return pmcid_str
    if pmcid_str.isdigit():
        return get_pmcid_from_pmid(pmcid_str)
    return None

In [ ]:
def process_single_article(pmcid):
    """Search for PRIDE IDs in a single article via BioC full-text API.

    Note: NCBI OA supplementary archive API (oa.fcgi) was retired in August 2026.
    Verification is now limited to BioC full-text only.
    """
    if not pmcid or not pmcid.startswith('PMC'):
        return set()

    bioc_url = f"https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_xml/{pmcid}/ascii"
    try:
        res = requests.get(bioc_url, timeout=15)
        if res.status_code == 200 and res.text:
            found = extract_pride_ids_from_content(res.text)
            if found:
                log.debug(f"{pmcid} BioC: {found}")
            return found
    except Exception:
        pass

    return set()

In [ ]:
def verify_and_update_pride_ids_range(input_csv, output_csv=None,
                                      start_row=0, end_row=None):
    """
    Verify articles from CSV in a given row range.
    For each article, search BioC + supplementary archives for additional PRIDE IDs.
    """
    if output_csv is None:
        name_parts = input_csv.replace('.csv', '')
        output_csv = f"{name_parts}_verified_rows_{start_row}_to_{end_row}.csv"

    log.info(f"Reading: {input_csv}")
    df_full = pd.read_csv(input_csv)
    log.info(f"Loaded {len(df_full)} records. Columns: {list(df_full.columns)}")

    if end_row is None:
        end_row = len(df_full)
    df_to_process = df_full.iloc[start_row:end_row].copy()
    log.info(f"Processing rows {start_row} to {end_row-1}: {len(df_to_process)} records")

    if len(df_to_process) == 0:
        log.warning("No records in specified range!")
        return None

    df_updated = df_full.copy()
    if 'PRIDE_ID_original' not in df_updated.columns:
        df_updated['PRIDE_ID_original'] = df_updated.get('PRIDE_ID', '').fillna('')
    if 'found_new_ids' not in df_updated.columns:
        df_updated['found_new_ids'] = False

    stats = {'total': len(df_to_process), 'updated': 0, 'new_ids_found': 0,
             'errors': 0, 'no_pmcid': 0, 'processed': 0}

    for idx, (original_idx, row) in enumerate(tqdm(df_to_process.iterrows(),
                                                    total=len(df_to_process),
                                                    desc="Verifying articles")):
        pmcid_str = row.get('PMCID', '')
        if pd.isna(pmcid_str):
            stats['no_pmcid'] += 1
            continue

        pmcid = parse_pmcid_from_string(pmcid_str)
        if not pmcid:
            stats['no_pmcid'] += 1
            continue

        current_ids = set()
        current_id_str = row.get('PRIDE_ID', '')
        if pd.notna(current_id_str) and str(current_id_str).strip() not in ['None Found', 'None', '', 'nan']:
            current_ids = set(id.strip() for id in str(current_id_str).split(',') if id.strip())

        try:
            found_ids = process_single_article(pmcid)
            stats['processed'] += 1

            if found_ids:
                all_ids = current_ids.union(found_ids)
                if all_ids != current_ids:
                    new_ids = all_ids - current_ids
                    log.info(f"  {pmcid}: NEW PRIDE IDs: {new_ids}")
                    df_updated.at[original_idx, 'PRIDE_ID'] = ', '.join(sorted(all_ids))
                    df_updated.at[original_idx, 'PRIDE_ID_verified'] = ', '.join(sorted(all_ids))
                    df_updated.at[original_idx, 'found_new_ids'] = True
                    stats['updated'] += 1
                    stats['new_ids_found'] += len(new_ids)
        except Exception as e:
            log.error(f"Error processing {pmcid}: {e}")
            stats['errors'] += 1

        time.sleep(0.5)

    df_updated.to_csv(output_csv, index=False, encoding='utf-8-sig')

    log.info(f"Verification complete — processed: {stats['processed']}, "
             f"no_pmcid: {stats['no_pmcid']}, updated: {stats['updated']}, "
             f"new_ids: {stats['new_ids_found']}, errors: {stats['errors']}")
    log.info(f"Saved to: {output_csv}")

    if stats['updated'] > 0:
        updated_only = df_updated[df_updated['found_new_ids'] == True]
        updated_file = output_csv.replace('.csv', '_only_updated.csv')
        updated_only.to_csv(updated_file, index=False, encoding='utf-8-sig')
        log.info(f"Updated records saved separately: {updated_file}")

    return df_updated

In [ ]:
# Example: verify rows 840-2641 of merged results
# result_df = verify_and_update_pride_ids_range(
#     input_csv="merged_pride_analysis_final.csv",
#     output_csv="pride_analysis_verified_rows_840_2641.csv",
#     start_row=839,
#     end_row=2641
# )

## 9. PRIDE ID Frequency Statistics

In [ ]:
def compute_pride_frequency(csv_path):
    """Compute PRIDE ID frequency table from a results CSV."""
    df = pd.read_csv(csv_path)
    all_ids = []
    for ids_str in df['PRIDE_ID'].dropna():
        ids_str = str(ids_str).strip()
        if ids_str not in ['None Found', 'None', '']:
            all_ids.extend(id.strip() for id in ids_str.split(',') if id.strip())

    id_counts = Counter(all_ids)
    log.info(f"Total records: {len(df)}")
    log.info(f"Unique articles (PMCID): {df['PMCID'].nunique()}")
    log.info(f"Unique PRIDE IDs: {len(id_counts)}")
    log.info(f"Total PRIDE ID mentions: {len(all_ids)}")
    log.info("Top-20 PRIDE IDs:")
    for pride_id, count in id_counts.most_common(20):
        log.info(f"  {pride_id}: {count}")

    stats_df = pd.DataFrame(id_counts.most_common(), columns=['PRIDE_ID', 'Frequency'])
    stats_df.to_csv("pride_id_frequency.csv", index=False, encoding='utf-8-sig')
    return stats_df

In [ ]:
# compute_pride_frequency("merged_pride_analysis_final.csv")